# Expected SARSA → DQN → Dyna-DQN

This notebook builds on our Expected SARSA experiment. We keep the sensor
observations, action choices, and 300 m route task, then change how the agent learns.
Run the cells from top to bottom using the project's `.venv` kernel.

| Agent | How it estimates action values | Where its practice comes from |
| --- | --- | --- |
| Expected SARSA | A weighted sum of features | The newest real step |
| DQN | A small neural network | Random batches of stored real steps |
| Dyna-DQN | The same Q network, plus a prediction network | Real steps plus predicted one-step outcomes |

A **Q value** estimates the total discounted reward from taking an action and
continuing afterward. It is an estimate, not a probability of avoiding a crash.

Implementation: `algorithms/dqn_agent.py`. Training loop: `experiments/dqn_helpers.py`.
A plain-language code walkthrough is in `algorithms/README.md`.

## 1. Setup

DQN needs PyTorch to build the networks and calculate weight updates. It is an
optional dependency; the original SARSA notebooks still work without it.
Install once from the project root if needed:

```bash
.venv/bin/python -m pip install -r requirements-dqn.txt
```

The agents run on the CPU to keep this example straightforward.

In [ ]:
from pathlib import Path
import sys, os, json
from dataclasses import asdict
from datetime import datetime
from zoneinfo import ZoneInfo
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "environments/sensor_highway/env.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside MORL4AVSO.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from environments.sensor_highway import ExperimentConfig
from algorithms.sensor_agent import AgentSettings
from algorithms.dqn_agent import DQNSettings, DQNAgent
from experiments.agent_helpers import train_agent, evaluate_agents
from experiments.dqn_helpers import train_dqn
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)

## 2. Reuse our observations, actions, and reward

`ObservationEncoder` makes the same small feature vector used by SARSA. It
summarizes nearby obstacle/free-space evidence, measurement confidence and age,
relative motion, ego telemetry, sensor status, and remaining route/time.
Unknown space is not treated as free road. No hidden traffic truth enters the networks.

There are **20 outputs**: five driving actions × four sensing modes (`off`,
`front`, `radar_rgb`, `all_on`). Each output is the Q estimate for one choice.
The network has one hidden layer of 64 units. A ReLU lets it learn relationships
that a single weighted sum cannot represent. It still loses details during encoding.

The reward matches the short-route SARSA notebook: progress earns reward, time
and sensing cost reward, a safe finish earns a bonus, and failures incur a penalty.
Keeping these settings fixed makes algorithm comparisons easier to interpret.

In [ ]:
TRAIN_SEED = 42
TRAIN_EPISODES = 60
TEST_SEEDS = list(range(100_000, 100_004))
TEST_WEATHER = ["clear", "rain", "fog"]

config = ExperimentConfig()
config.highway.update(duration=16, high_speed_reward=0.0, collision_reward=-10.0)
config.weather.preset = "random"
config.weather.severity = 0.7
config.reward.route_distance_m = 300
config.reward.progress_weight = 0.02
config.reward.time_weight = 0.1
config.reward.completion_bonus = 10.0
config.reward.early_completion_bonus = 5.0
config.reward.failure_penalty = 10.0
config.validate()

settings = DQNSettings(episodes=TRAIN_EPISODES)
sarsa_settings = AgentSettings(episodes=TRAIN_EPISODES, alpha=0.12, gamma=settings.gamma,
                              epsilon_start=settings.epsilon_start, epsilon_end=settings.epsilon_end)
assert not set(TEST_SEEDS).intersection(range(TRAIN_SEED, TRAIN_SEED + TRAIN_EPISODES))
display(pd.Series(asdict(settings), name="DQN settings"))

## 3. Understand the DQN parts

**Choosing an action (`select`).** With probability epsilon, choose randomly.
Otherwise, choose the highest Q value. Epsilon decreases from 1.0 to 0.1 during training.
This exploration rule is the same idea as our SARSA agent.

**Replay buffer (`ReplayBuffer`).** Save each real step as
`(features, action, reward, next_features, terminal)`. Randomly sample 32 stored
steps at a time. Mixing experience reduces dependence on consecutive road scenes.
Wait for 64 stored steps before starting Q updates. The buffer holds at most 10,000.

**Two Q networks (`online` and `target`).** The online network learns every update.
The target network supplies future-value estimates and is copied from the online
network every 100 Q updates. This makes the training target change less often.

**Learning target (`td_targets`).** SARSA averages future Q values under its
exploring policy. DQN instead uses the best estimated next action:

```text
SARSA: target = reward + gamma × [(1-epsilon) × max(Q_next) + epsilon × mean(Q_next)]
DQN:   target = reward + gamma × max(target_network(next_features))
```

A crash, off-road ending, route finish, or route deadline has `target = reward`.
For the original continuing-driving task, an artificial time cutoff still includes
future value, matching our existing SARSA convention.

**Weight update (`learn_batch`).** Select the online Q output for the action that
was taken, compare it with the target using Smooth L1 loss, and let Adam adjust
the weights. Smooth L1 reduces the influence of unusually large errors. Gradient
clipping limits the size of the gradient before the optimizer uses it.
These are standard DQN ingredients; see the
[official PyTorch DQN tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html).
The code here uses ordinary DQN with periodic target copies.

In [ ]:
dqn_agent, dqn_history = train_dqn(config, settings, seed=TRAIN_SEED)
print(dqn_agent.online)
display(dqn_history.tail())

## 4. Add the Dyna offshoot

Dyna adds a second way to practice: predict what might happen, then learn from
that prediction. **Replay alone is not Dyna**: replay repeats actual experience;
Dyna uses a model to produce outcomes that were not directly observed.

**Prediction model (`model_predictions`).** Input the current features and action.
The action is represented by a vector with one entry set to 1. Output a predicted
feature change, reward, and probability that no future value should follow.
Predicting the change means the model learns what moved rather than rebuilding
the entire observation from scratch.

**Model learning (`learn_model`).** Compare predictions against real stored steps.
Use Smooth L1 for feature changes and rewards, and binary cross-entropy for the
terminal flag. Train the model once per real Q update. It never receives simulator
traffic truth.

**Planning (`imagine_batch`).** After 200 stored real steps, sample real starting
feature/action pairs and predict their outcomes. Keep features within [-1, 1] and
preserve the encoder's constant bias. Use the prediction to make two extra Q
updates per real step:

```text
imagined_target = predicted_reward
                 + gamma × (1 - predicted_terminal_probability)
                         × max(target_network(predicted_next_features))
```

We use the ending probability directly instead of forcing a yes/no decision.
Each prediction starts from real data and goes only one step ahead. Imagined
transitions are never stored in the real replay buffer, and Q updates do not
backpropagate into the prediction model.

This is a simple Dyna-style feature model. Predictions can be wrong, especially
for rare crashes or partially observed traffic. Clipping does not guarantee a
physically valid scene. Planning may hurt performance; it is a hypothesis to test.
Set `planning_steps=0` to disable imagined updates. Dyna uses more computation
than DQN at the same real episode budget.

In [ ]:
dyna_agent, dyna_history = train_dqn(config, settings, seed=TRAIN_SEED, dyna=True)
display(dyna_history.tail())

## 5. Train a fresh Expected SARSA reference

Use the same reward, features, exploration schedule, episode count, and seed range.
The algorithms can choose different actions, so they collect different experience.
Equal episode counts also do not guarantee equal numbers of real steps because
crashes and early finishes change episode length. Compare `steps` as well.
The neural learning rate and SARSA alpha have different meanings.

In [ ]:
sarsa_agent, sarsa_history = train_agent(config, sarsa_settings, seed=TRAIN_SEED)
histories = {"Expected SARSA": sarsa_history, "DQN": dqn_history, "Dyna-DQN": dyna_history}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for name, history in histories.items():
    for ax, metric in zip(axes, ["return", "crashed", "energy_j"]):
        ax.plot(history.episode, history[metric].astype(float).rolling(10, min_periods=1).mean(), label=name)
        ax.set(xlabel="Episode", title=metric.replace("_", " "))
        ax.grid(alpha=.2)
axes[0].legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame({name: {"real_steps": int(h.steps.sum())} for name, h in histories.items()}).T)

## 6. Evaluate without learning or exploration

The existing evaluator already works with all three agents: each supports
`select`, `decode`, and `feature_schema`. It calls no learning methods.
Use new traffic seeds in clear, rain, and fog conditions. Include the existing
fixed IDLE and random-driving baselines with all sensors on.

Read completion and crash fractions first, then successful completion time and
sensing power. An early crash can produce low total energy. Low training loss
only means the network fits its targets; it does not show good driving.
Sixty episodes and four seeds per weather are an exploratory starting point.
Repeat training seeds before claiming that one algorithm is better.

In [ ]:
agents = {"Expected SARSA": sarsa_agent, "DQN": dqn_agent, "Dyna-DQN": dyna_agent,
          "idle_all_on": None, "random_all_on": None}
evaluation = evaluate_agents(config, agents, TEST_SEEDS, TEST_WEATHER)
summary = evaluation.groupby("policy", sort=False).agg(
    episodes=("seed", "count"), mean_return=("return", "mean"),
    crash_fraction=("crashed", "mean"), completion_fraction=("completed", "mean"),
    successful_finish_time_s=("completion_time_s", "mean"),
    mean_power_w=("mean_power_w", "mean"), mean_distance_m=("distance_m", "mean"))
display(summary.round(3))

## 7. Save models and matching settings

Save policies with their environment configuration, histories, and evaluation.
`DQNAgent.load` restores either neural policy for evaluation. It does **not**
restore the replay buffer, optimizer, target-update count, or learned dynamics
model, so this is not an exact training-resume checkpoint. Use the saved config
with its matching observation encoder. Each save creates a fresh directory.

In [ ]:
SAVE_RUN = True
if SAVE_RUN:
    stamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y%m%d_%H%M%S_%f")
    output = PROJECT_ROOT / "runs/comparisons" / f"dqn_comparison_{stamp}"
    output.mkdir(parents=True, exist_ok=False)
    config.save(output / "config.json")
    dqn_agent.save(output / "dqn.pt")
    dyna_agent.save(output / "dyna_dqn.pt")
    sarsa_agent.save(output / "sarsa.npz")
    for name, history in [("dqn", dqn_history), ("dyna_dqn", dyna_history), ("sarsa", sarsa_history)]:
        history.to_csv(output / f"{name}_training.csv", index=False)
    evaluation.to_csv(output / "evaluation.csv", index=False)
    summary.to_csv(output / "summary.csv")
    (output / "metadata.json").write_text(json.dumps({
        "dqn_settings": asdict(settings), "sarsa_settings": asdict(sarsa_settings),
        "training_seed": TRAIN_SEED, "test_seeds": TEST_SEEDS, "test_weather": TEST_WEATHER,
        "feature_schema": dqn_agent.feature_schema, "actions": dqn_agent.actions,
        "torch": str(torch.__version__), "numpy": np.__version__, "python": sys.version,
        "checkpoint_purpose": "evaluation only",
    }, indent=2))
    restored = DQNAgent.load(output / "dyna_dqn.pt")
    for original, loaded in zip(dyna_agent.online.parameters(), restored.online.parameters()):
        assert torch.equal(original, loaded)
    print("Saved:", output)

## 8. A simple next experiment

Run DQN, then Dyna-DQN with `planning_steps` of 0, 1, and 2. Keep the task and real
interaction budget fixed and repeat several training seeds. Compare completion,
crashes, power, real steps, and total Q updates. If planning hurts, the learned
model may need better data or more time before it is used.

The shared encoder discards scene detail and the agents have no recurrent memory.
A deterministic one-step model cannot fully describe uncertain hidden traffic.
This example implements the algorithms; its performance must be measured.